In [3]:
import ee
import pandas as pd
from google.colab import files # Librería exclusiva de Colab para descargas

# 1. Autenticación e Inicialización en Colab
# Esto abrirá una ventana emergente para que inicies sesión con tu cuenta de Google
ee.Authenticate()

# REEMPLAZA ESTO con tu ID de proyecto de Google Cloud
ee.Initialize(project='estadistica-earth')

# 2. Definir la Región de Interés (ROI)
venezuela = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq('ADM0_NAME', 'Venezuela'))

# 3. Cargar el dataset VIIRS
viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
          .select('avg_rad') \
          .filterDate('2006-01-01', '2024-01-01')

# 4. Función para calcular el promedio
def get_mean_radiance(image):
    mean_dict = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=venezuela.geometry(),
        scale=500,
        maxPixels=1e9
    )
    date = image.date().format('YYYY-MM-dd')
    return ee.Feature(None, {
        'date': date,
        'mean_radiance': mean_dict.get('avg_rad')
    })

# 5. Procesamiento
print("Procesando imágenes en los servidores de Google... Esto puede tomar un minuto.")
radiance_features = viirs.map(get_mean_radiance)

# 6. Extracción de datos
data_list = radiance_features.reduceColumns(
    ee.Reducer.toList(2), ['date', 'mean_radiance']
).values().get(0).getInfo()

# 7. Estructurar con Pandas
df = pd.DataFrame(data_list, columns=['Fecha', 'Radiancia_Media'])
df['Fecha'] = pd.to_datetime(df['Fecha'])

# 8. Exportar y Descargar
filename = 'radiancia_nocturna_venezuela_viirs.csv'
df.to_csv(filename, index=False)
print(f"Extracción completada. Descargando {filename}...")

# Comando de Colab para forzar la descarga del archivo a tu disco duro local
files.download(filename)

Procesando imágenes en los servidores de Google... Esto puede tomar un minuto.
Extracción completada. Descargando radiancia_nocturna_venezuela_viirs.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
import ee
import pandas as pd
from google.colab import files

# 1. Autenticación e Inicialización
ee.Authenticate()
ee.Initialize(project='estadistica-earth') # RECUERDA PONER TU ID AQUÍ

print("Construyendo geometrías...")

# 2. Definir las Regiones de Interés (Nacional + 3 Estados)
# Extraemos el polígono nacional
venezuela = ee.FeatureCollection("FAO/GAUL/2015/level0") \
    .filter(ee.Filter.eq('ADM0_NAME', 'Venezuela')) \
    .map(lambda f: f.set('Region', 'Nacional')) # Le ponemos etiqueta manual

# Extraemos los 3 estados
estados = ee.FeatureCollection("FAO/GAUL/2015/level1") \
    .filter(ee.Filter.inList('ADM1_NAME', ['Carabobo', 'Zulia', 'Miranda'])) \
    .map(lambda f: f.set('Region', f.get('ADM1_NAME'))) # Usamos el nombre del estado

# Unimos todo en una sola colección de 4 polígonos
roi_combinada = venezuela.merge(estados)

# =====================================================================
# BLOQUE 1: EXTRACCIÓN SATÉLITE MODERNO (VIIRS - MENSUAL - RADIANCIA)
# =====================================================================
print("Extrayendo datos de VIIRS (2014-2024)... Esto tomará un par de minutos.")

viirs = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG") \
          .select('avg_rad') \
          .filterDate('2014-01-01', '2024-01-01')

def get_viirs_data(image):
    means = image.reduceRegions(collection=roi_combinada, reducer=ee.Reducer.mean(), scale=500)
    date = image.date().format('YYYY-MM-dd')
    return means.map(lambda f: f.set('Fecha', date))

viirs_features = viirs.map(get_viirs_data).flatten()
viirs_data = viirs_features.reduceColumns(ee.Reducer.toList(3), ['Fecha', 'Region', 'mean']).values().get(0).getInfo()

df_viirs = pd.DataFrame(viirs_data, columns=['Fecha', 'Region', 'Radiancia_VIIRS'])
df_viirs['Fecha'] = pd.to_datetime(df_viirs['Fecha'])
df_viirs.to_csv('viirs_moderno_mensual.csv', index=False)


# =====================================================================
# BLOQUE 2: EXTRACCIÓN SATÉLITE HISTÓRICO (DMSP - ANUAL - LÍNEA BASE)
# =====================================================================
print("Extrayendo datos históricos de DMSP (1992-2013)...")

dmsp = ee.ImageCollection("NOAA/DMSP-OLS/NIGHTTIME_LIGHTS") \
         .select('stable_lights') \
         .filterDate('1992-01-01', '2014-01-01')

def get_dmsp_data(image):
    # La escala es 1000m porque este satélite tenía menor resolución
    means = image.reduceRegions(collection=roi_combinada, reducer=ee.Reducer.mean(), scale=1000)
    # Como es anual, solo extraemos el año
    year = image.date().format('YYYY')
    return means.map(lambda f: f.set('Anio', year))

dmsp_features = dmsp.map(get_dmsp_data).flatten()
dmsp_data = dmsp_features.reduceColumns(ee.Reducer.toList(3), ['Anio', 'Region', 'mean']).values().get(0).getInfo()

df_dmsp = pd.DataFrame(dmsp_data, columns=['Anio', 'Region', 'Intensidad_DMSP_0_a_63'])
df_dmsp.to_csv('dmsp_historico_anual.csv', index=False)

# =====================================================================
# DESCARGA DE ARCHIVOS
# =====================================================================
print("¡Extracción completada! Descargando archivos...")
files.download('viirs_moderno_mensual.csv')
files.download('dmsp_historico_anual.csv')

Construyendo geometrías...
Extrayendo datos de VIIRS (2014-2024)... Esto tomará un par de minutos.
Extrayendo datos históricos de DMSP (1992-2013)...
¡Extracción completada! Descargando archivos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>